<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Pipeline - Decision Tree Regression - Diamonds
</b></font> </br></p>

---

# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten & Strukturen
from pandas import read_csv, DataFrame  # CSV laden und Daten verwalten

# Datenvorverarbeitung
from sklearn.preprocessing import OrdinalEncoder  # Encoding

# Pipelines & Transformation
from sklearn.pipeline import Pipeline  # Verarbeitungsschritte kombinieren
from sklearn.compose import ColumnTransformer  # Spaltenspezifische Transformationen

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell
from sklearn.tree import DecisionTreeRegressor  # Regressionsmodell

# Evaluation & Metriken
from sklearn.metrics import r2_score, mean_absolute_error  # Regressionsmetriken

# Visualisierung
import plotly.express as px  # Interaktive Plots

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
***

<font color='black' size="5">📋 Checkliste</font>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<font color='black' size="5">
Anwendungsfall
</font>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)

In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[ "carat", "cut", "color", "clarity", "depth", "table", "price", ])

In [ ]:
data = df.copy()
target = data.pop("price")

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<font color='black' size="5">
Datentyp ermitteln
</font>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<font color='black' size="5">
Train-Test-Split
</font>

In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.30, shuffle=True, random_state=42)

<font color='black' size="5">
Kodierung - Pipeline für kategoriale Features
</font>

<p><font color='black' size="5">
Modellbezogene Vorverarbeitung
</font></p>

Für Decision Tree sind Skalierung und Imputation in diesem Datensatz nicht erforderlich. Die Pipeline kodiert nur die kategorialen Features; numerische Features werden unverändert weitergegeben.

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

pipe_cat = Pipeline(
    [
        ("encoder", OrdinalEncoder(categories=cat_seq)),
    ]
)

<font color='black' size="5">
Numerische Features unverändert weitergeben
</font>

In [ ]:
pipe_num = "passthrough"


<font color='black' size="5">
Verbinden der beiden Pipelines zu einer Prepare-Pipeline
</font>

In [ ]:
pipe_prepare = ColumnTransformer(
    transformers=[
        ("categorical", pipe_cat, cat_col),
        ("numerical", pipe_num, num_col)
    ]
)

# 3 |  Modeling
---

 <font color='black' size="5">
Modellauswahl, Verbinden Prepare- & Model-Pipeline, Training
</font>

<font color='black' size="5">📋 Checkliste</font>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

**Wann sklearn Pipeline?**

**In der Praxis relevant wenn:**
- Preprocessing und Modelltraining zusammen als eine Einheit verwaltet werden sollen — verhindert Data Leakage
- Das gleiche Workflow-Objekt für Training und Inferenz verwendet werden soll (`.fit()` → `.predict()` ohne Zwischenschritte)
- Hyperparameter-Tuning über Preprocessing und Modell gleichzeitig erfolgen soll (Pipeline ist direkt mit GridSearch kombinierbar)

**Nicht geeignet wenn:**
- Sehr individuelles Preprocessing erforderlich ist, das sich nicht in Standard-Transformatoren abbilden lässt
- Nur ein einziger Preprocessing-Schritt existiert → dann ist eine Pipeline Overhead ohne Mehrwert

**Typischer Fehler:**
Preprocessing außerhalb der Pipeline auf allen Daten (inklusive Testdaten) anwenden und dann die vorverarbeiteten Daten in die Pipeline geben — das ist der häufigste Weg zu Data Leakage und verfälscht die Evaluierungsergebnisse.

In [ ]:
# Create a separate pipeline for the model
pipe_model = Pipeline(
    [
        ("tree", DecisionTreeRegressor(max_depth=4, min_samples_split=50, random_state=42))
    ])

In [ ]:
# Combine the prepare and model pipelines
model = Pipeline(
    [
        ("prepare", pipe_prepare),
        ("model", pipe_model)
    ])

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<font color='black' size="5">📋 Checkliste</font>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<font color='black' size="5">
Prediction
</font>

In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<font color='black' size="5">
Bestimmtheitsmass
</font>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"-- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"-- Test --- Bestimmtheitsmass: {r2:5.2f}")

<font color='black' size="5">
Mean Absolut Error
</font>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"-- Test -- Mean Absolute Error: {mae:5.2f}")

<font color='black' size="5">
Analyse von Zwischenschritten einer Pipeline
</font>

In [ ]:
# Konfiguration Pipeline 'prepare'
model.named_steps.prepare

In [ ]:
# Beispiel: vorbereitete Feature-Matrix für die ersten Trainingszeilen
model.named_steps["prepare"].transform(data_train.head())


In [ ]:
# Anzahl der vom Modell verwendeten Features
model.named_steps["model"].named_steps["tree"].n_features_in_

<font color='black' size="5">
Feature Importance
</font>

In [ ]:
# Feature Importance vom Modell
importance = model.named_steps["model"].named_steps["tree"].feature_importances_
print("Feature Importance Values:")
for i, imp in enumerate(importance):
    print(f"Feature {i}: {imp:.4f}")

In [ ]:
# Feature Namen zuordnen
# Nach ColumnTransformer: [categorical features] + [numerical features]
feature_names = list(cat_col) + list(num_col)
print("Feature Names nach Transformation:")
print(feature_names)

# Feature Importance mit Namen
feature_importance_df = DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (sortiert):")
print(feature_importance_df)

In [ ]:
# Alternative Visualisierung mit Plotly
fig = px.bar(feature_importance_df,
             x='Importance',
             y='Feature',
             orientation='h',
             title='Decision Tree - Feature Importance',
             labels={'Importance': 'Feature Importance', 'Feature': 'Features'})
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# 5 | Deploy
---

<font color='black' size="5">📋 Checkliste</font>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>